# Solutions — Maps, Structs & Complex Types (Medium 15)

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_transactions = spark.table("samples.bakehouse.sales_transactions")
df_franchises   = spark.table("samples.bakehouse.sales_franchises")
df_clickstream  = spark.table("samples.wanderbricks.clickstream")

## Solution 1 — Create Map Column and Extract by Key

In [ ]:
result_1 = (
    df_transactions
    .withColumn("transaction_metadata",
        F.create_map(
            F.lit("product"),  F.col("product"),
            F.lit("payment"),  F.col("paymentMethod")
        ))
    .withColumn("product_from_map", F.col("transaction_metadata")["product"])
    .select("transactionID","transaction_metadata","product_from_map")
)

## Solution 2 — map_from_entries: Revenue Map per Franchise

In [ ]:
# Step 1: Pre-aggregate total revenue per franchise + product (avoids nested aggregates)
product_revenue = (
    df_transactions
    .join(df_franchises, on="franchiseID")
    .groupBy("franchiseID", "name", "product")
    .agg(F.round(F.sum("totalPrice"), 2).alias("revenue"))
)

# Step 2: Collect the pre-aggregated rows into a map per franchise
result_2 = (
    product_revenue
    .groupBy("franchiseID", "name")
    .agg(
        F.map_from_entries(
            F.collect_list(F.struct(F.col("product"), F.col("revenue")))
        ).alias("product_revenue_map")
    )
    .withColumn("product_count", F.size("product_revenue_map"))
    .orderBy("franchiseID")
)

## Solution 3 — Create Struct Column, Access Nested Fields

In [ ]:
result_3 = (
    df_franchises
    .withColumn("location",
        F.struct(F.col("city").alias("city"), F.col("country").alias("country")))
    .withColumn("city_from_struct", F.col("location.city"))
    .select("franchiseID","name","location","city_from_struct")
)

## Solution 4 — Extract Struct Fields from Clickstream Metadata

In [ ]:
# metadata is already a struct<device:string, referrer:string>, not a JSON string
# Access sub-fields directly via dot notation
result_4 = (
    df_clickstream
    .select(
        F.col("metadata.device").alias("device"),
        F.col("event"),
    )
    .groupBy("device", "event")
    .agg(F.count("*").alias("event_count"))
    .orderBy(F.col("event_count").desc())
)

## Solution 5 — Map Keys Extraction to Array

In [ ]:
# Step 1: Pre-aggregate total revenue per franchise + paymentMethod
payment_revenue = (
    df_transactions
    .groupBy("franchiseID", "paymentMethod")
    .agg(F.round(F.sum("totalPrice"), 2).alias("revenue"))
)

# Step 2: Collect into a map per franchise
result_5 = (
    payment_revenue
    .groupBy("franchiseID")
    .agg(
        F.map_from_entries(
            F.collect_list(F.struct(
                F.col("paymentMethod").alias("key"),
                F.col("revenue").alias("value")
            ))
        ).alias("revenue_by_payment")
    )
    .withColumn("payment_methods", F.map_keys("revenue_by_payment"))
    .select("franchiseID", "revenue_by_payment", "payment_methods")
)

## Solution 6 — Explode Map Back to Rows

In [ ]:
# Rebuild the payment revenue map (same approach as P5, pre-aggregated)
payment_revenue_agg = (
    df_transactions
    .groupBy("franchiseID", "paymentMethod")
    .agg(F.round(F.sum("totalPrice"), 2).alias("revenue"))
)

franchise_map = (
    payment_revenue_agg
    .groupBy("franchiseID")
    .agg(
        F.map_from_entries(
            F.collect_list(F.struct(
                F.col("paymentMethod").alias("key"),
                F.col("revenue").alias("value")
            ))
        ).alias("revenue_by_payment")
    )
)

result_6 = (
    franchise_map
    .select("franchiseID", F.explode("revenue_by_payment").alias("payment_method", "revenue"))
    .orderBy("franchiseID", "payment_method")
)

## Solution 7 — JSON String to Struct, Extract Field

In [ ]:
schema = T.StructType([
    T.StructField("customerID", T.StringType()),
    T.StructField("country",    T.StringType()),
])
result_7 = (
    df_transactions
    .withColumn("json_payload",
        F.to_json(F.struct(F.col("customerID"), F.lit("unknown").alias("country"))))
    .withColumn("parsed", F.from_json("json_payload", schema))
    .withColumn("extracted_country", F.col("parsed.country"))
    .select("customerID","json_payload","extracted_country")
    .limit(100)
)